# ADVC W1b: INT8 probes (branch `revision-w1`, Kaggle T4)

Follows up on W1 (git 89018eb): the INT8 classifier head (200 outputs) raised `cublasLt ran into an error` at batch 32 while the batch-1 dummy forward worked, and reloading a saved INT8 state_dict raised after the first forward.

**Before running:** GPU T4 x2, Internet On, Add Input **jainmanav20/advc-results** only (no image dataset needed; inputs are random tensors and the July checkpoints).

Then **Save Version -> Save & Run All (Commit)**. Estimate (unmeasured): ~5-8 min wall, of which ~2-3 min GPU. Copy the output of the **last cell** back to Claude.

| Cell | What |
|---|---|
| P1 | Which (out_features, batch) shapes does bnb 0.50.2 Linear8bitLt run on a T4? forward + backward-to-input |
| P2 | Is padding the 200-way head to a working width numerically identical to quantizing the 200-way head? |
| P3 | Does the new loader path (load INT8 state_dict before the first forward) load a July INT8 checkpoint exactly? |
| T | Re-run tests/test_runlog.py (Wilson fix) |

In [ ]:
# 1. Setup: clone branch, pin bitsandbytes, link checkpoints, helper
import os, subprocess, sys, time, glob, json
BRANCH = 'revision-w1'
W = '/kaggle/working/w1b'
os.makedirs(W, exist_ok=True)
if not os.path.isdir('/kaggle/working/ADVC'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/Jmanav/ADVC.git', '/kaggle/working/ADVC'], check=True)
os.chdir('/kaggle/working/ADVC')
GIT = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout.strip()
print('git:', GIT)
!pip install -q --no-deps bitsandbytes==0.50.2
!pip install -q torchattacks pytest
import torch, bitsandbytes
print('torch', torch.__version__, '| bitsandbytes', bitsandbytes.__version__, '|', torch.cuda.get_device_name(0))
assert bitsandbytes.__version__ == '0.50.2'
hits = glob.glob('/kaggle/input/**/at_int8_epoch01.pt', recursive=True)
assert hits, 'No checkpoints found: attach jainmanav20/advc-results'
CKPT_ROOT = os.path.dirname(os.path.dirname(hits[0]))
os.makedirs('results/checkpoints', exist_ok=True)
if not os.path.exists('results/checkpoints/tiny-imagenet'):
    os.symlink(CKPT_ROOT, 'results/checkpoints/tiny-imagenet')
assert os.path.isfile('results/checkpoints/tiny-imagenet/base/deit_small_ft.pt')

TIMES = {}
def sh(name, cmd, env=None):
    t0 = time.time()
    with open(f'{W}/{name}.log', 'w') as log:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, **(env or {})))
        for line in p.stdout:
            log.write(line); print(line, end='')
        p.wait()
    TIMES[name] = {'seconds': round(time.time() - t0, 1), 'exit': p.returncode}
    print('[' + name + ']', TIMES[name])
    return p.returncode

In [ ]:
%%writefile /kaggle/working/w1b/p1_shapes.py
# P1: one process per out_features N (a CUDA error could poison the context); loops batch sizes.
import json, sys, traceback
sys.path.insert(0, '.')
import torch
from models.loader import replace_linear_int8

N = int(sys.argv[1])
BS = [1, 2, 4, 8, 15, 16, 17, 31, 32, 33, 64]
out = {}
torch.manual_seed(0)
m = torch.nn.Sequential(torch.nn.Linear(384, N))
replace_linear_int8(m)
m = m.cuda()
for B in BS:
    x = torch.randn(B, 384, device='cuda', requires_grad=True)
    try:
        y = m(x)
        g, = torch.autograd.grad(y.float().sum(), x)
        torch.cuda.synchronize()
        out[B] = 'ok' if torch.isfinite(y).all() and torch.isfinite(g).all() else 'nonfinite'
    except Exception as e:
        out[B] = type(e).__name__ + ': ' + str(e).splitlines()[0][:60]
        try:
            torch.ones(2, device='cuda').sum().item()
            out[str(B) + '_ctx'] = 'cuda context ok after error'
        except Exception as e2:
            out[str(B) + '_ctx'] = 'cuda context BROKEN: ' + str(e2)[:60]
print('P1JSON ' + json.dumps({'N': N, 'res': out}))

In [ ]:
# P1 run: out_features 200 (TinyImageNet head), nearby multiples of 8/16/32, and 1000 (Imagenette head)
P1 = {}
for N in [200, 208, 216, 224, 232, 240, 256, 384, 1000, 1024]:
    r = subprocess.run([sys.executable, f'{W}/p1_shapes.py', str(N)], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith('P1JSON ')]
    P1[N] = json.loads(line[0][7:])['res'] if line else {'error': (r.stdout + r.stderr)[-600:]}
    print(N, P1[N])
json.dump(P1, open(f'{W}/p1_shapes.json', 'w'), indent=1)

In [ ]:
%%writefile /kaggle/working/w1b/p2_pad.py
# P2: LLM.int8 quantizes weights per output row and activations per input row, so padding the head with
# extra rows and slicing the logits SHOULD give identical numbers. Verify on random weights.
import json, sys
sys.path.insert(0, '.')
import torch
from models.loader import replace_linear_int8

P = int(sys.argv[1])
torch.manual_seed(0)
W = torch.randn(200, 384) * 0.02
b = torch.randn(200) * 0.01
Wp = torch.cat([W, W[: P - 200]])      # pad with copies of real rows (avoids all-zero rows)
bp = torch.cat([b, b[: P - 200]])

def mk(w, bb):
    lin = torch.nn.Linear(384, w.shape[0])
    lin.weight.data = w.clone(); lin.bias.data = bb.clone()
    s = torch.nn.Sequential(lin); replace_linear_int8(s)
    return s.cuda()

m200, mP = mk(W, b), mk(Wp, bp)
r = {'P': P}
def safe_eq(a, b):
    try:
        return bool(torch.equal(a, b))
    except Exception as e:
        return 'n/a: ' + repr(e)[:80]
r['CB_rows_equal'] = safe_eq(m200[0].weight.data, mP[0].weight.data[:200])
r['SCB_rows_equal'] = safe_eq(getattr(m200[0].weight, 'SCB', None), (getattr(mP[0].weight, 'SCB', None) if getattr(mP[0].weight, 'SCB', None) is None else mP[0].weight.SCB[:200]))
x = torch.randn(32, 384)
ref = x @ W.T + b
with torch.no_grad():
    o200_rows = torch.cat([m200(x[i:i + 1].cuda()) for i in range(32)]).float().cpu()
    oP_rows = torch.cat([mP(x[i:i + 1].cuda())[:, :200] for i in range(32)]).float().cpu()
    oP = mP(x.cuda())[:, :200].float().cpu()
r['max_abs_200_vs_padded_bs1'] = float((o200_rows - oP_rows).abs().max())
r['max_abs_padded_bs32_vs_bs1'] = float((oP - oP_rows).abs().max())
r['max_abs_padded_bs32_vs_fp32_ref'] = float((oP - ref).abs().max())
r['max_abs_200_bs1_vs_fp32_ref'] = float((o200_rows - ref).abs().max())
r['argmax_agree_padded_vs_200'] = float((oP.argmax(1) == o200_rows.argmax(1)).float().mean())
print('P2JSON ' + json.dumps(r))

In [ ]:
# P2 run for candidate padded widths (the first that P1 shows working at batch 16 and 32 is the one that matters)
P2 = {}
for P in [208, 224, 256]:
    r = subprocess.run([sys.executable, f'{W}/p2_pad.py', str(P)], capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith('P2JSON ')]
    P2[P] = json.loads(line[0][7:]) if line else {'error': (r.stdout + r.stderr)[-800:]}
    print(P, P2[P])
json.dump(P2, open(f'{W}/p2_pad.json', 'w'), indent=1)

In [ ]:
%%writefile /kaggle/working/w1b/p3_reload.py
# P3: load a July INT8 AT checkpoint through load_model(int8_state_dict=...) (before the first forward).
import json, sys, traceback
sys.path.insert(0, '.')
import torch
from models.loader import load_config, load_model

CK = 'results/checkpoints/tiny-imagenet/at/at_int8_epoch07.pt'
cfg = load_config('configs/base.yaml')
sd = torch.load(CK, map_location='cuda')
r = {'checkpoint': CK}
try:
    m = load_model('deit_small', 'int8', cfg, device='cuda', int8_state_dict=sd)
    r['load'] = 'OK'
    r['load_result'] = m.__dict__.pop('_advc_int8_load', None)
except Exception:
    r['load'] = 'RAISED'
    r['traceback'] = traceback.format_exc()[-3000:]
    print('P3JSON ' + json.dumps(r)); sys.exit(0)
same_w = diff_w = same_b = diff_b = 0
for name, mod in m.named_modules():
    if type(mod).__name__ != 'Linear8bitLt':
        continue
    w_saved = sd[name + '.weight']
    st = getattr(mod, 'state', None)
    w_now = st.CB if (st is not None and getattr(st, 'CB', None) is not None) else mod.weight.data
    if torch.equal(w_now.to(w_saved.dtype), w_saved): same_w += 1
    else: diff_w += 1
    if mod.bias is not None and (name + '.bias') in sd:
        if torch.equal(mod.bias.detach().float(), sd[name + '.bias'].float()): same_b += 1
        else: diff_b += 1
r['int8_weights_equal_saved'] = [same_w, diff_w]
r['biases_equal_saved'] = [same_b, diff_b]
ln_keys = [k for k in sd if 'norm' in k]
r['norm_params_equal_saved'] = sum(int(torch.equal(m.state_dict()[k].float(), sd[k].float())) for k in ln_keys)
r['n_norm_keys'] = len(ln_keys)
torch.manual_seed(0)
x = torch.randn(1, 3, 224, 224, device='cuda')
with torch.no_grad():
    out = m(x)
    out = out.logits if hasattr(out, 'logits') else out
    base = load_model('deit_small', 'int8', cfg, device='cuda')
    ob = base(x); ob = ob.logits if hasattr(ob, 'logits') else ob
r['logits_finite'] = bool(torch.isfinite(out).all())
r['max_abs_logit_diff_vs_untrained_base'] = float((out.float() - ob.float()).abs().max())
print('P3JSON ' + json.dumps(r))

In [ ]:
# P3 run (~1 min) and the Wilson-fix test
sh('p3_reload', [sys.executable, f'{W}/p3_reload.py'])
sh('pytest_runlog', [sys.executable, '-m', 'pytest', '-q', 'tests/test_runlog.py'])

In [ ]:
# SUMMARY: copy everything this cell prints back to Claude (also saved to /kaggle/working/w1b/summary.txt). Not truncated.
lines = ['git: ' + GIT, 'torch ' + torch.__version__ + ' | bnb ' + bitsandbytes.__version__ + ' | ' + torch.cuda.get_device_name(0)]
lines.append('== P1: Linear8bitLt(384, N) forward+backward by batch size ==')
BS = ['1', '2', '4', '8', '15', '16', '17', '31', '32', '33', '64']
lines.append('N     ' + ' '.join(b.rjust(4) for b in BS))
for N, res in P1.items():
    if 'error' in res:
        lines.append(str(N).ljust(6) + 'SCRIPT ERROR: ' + res['error'])
        continue
    cells = [('ok' if res.get(b) == 'ok' else 'X') .rjust(4) for b in BS]
    lines.append(str(N).ljust(6) + ' '.join(cells))
    errs = sorted({v for k, v in res.items() if v != 'ok'})
    for e in errs:
        lines.append('      ' + e)
lines.append('== P2: padded head equivalence ==')
for P, r in P2.items():
    lines.append(json.dumps(r))
lines.append('== P3: INT8 reload via load_model(int8_state_dict=...) ==')
for l in open(f'{W}/p3_reload.log'):
    if l.startswith('P3JSON '):
        lines.append(json.dumps(json.loads(l[7:]), indent=1))
lines.append('== pytest_runlog (last 15 lines) ==')
lines += [l.rstrip() for l in open(f'{W}/pytest_runlog.log').readlines()[-15:]]
lines.append('== wall times ==')
lines.append(json.dumps(TIMES))
text = '\n'.join(lines)
open(f'{W}/summary.txt', 'w').write(text)
print(text)